# 1つの式では、ストライドとスライスを同時に使わない

Python には、somelist[start:end:stride]という形式でスライスの増分（ストライド）を規定する構文があります。

In [1]:
x = ['red', 'green', 'blue', 'yellow', 'purple', 'orange']
odds = x[::2]
evens = x[1::2]
print("Odd indexed colors:", odds)
print("Even indexed colors:", evens)

Odd indexed colors: ['red', 'blue', 'purple']
Even indexed colors: ['green', 'yellow', 'orange']


ただ問題があります。このストライド構文はバグをもたらす予期せぬ振る舞いをすることがあるのです。

例えば、よくある Python での巧妙な技法で、バイト列を逆転するのには、増分を -1　にしてバイト列をスライスします。

In [2]:
x = b'mongoose'
y = x[::-1]
print(y)  # 出力: b'esoonogm'

b'esoognom'


これは、Unicode 文字列でも正しく動作します。

In [3]:
x = '寿司'
y = x[::-1]
print(y)  # 出力: '司寿'

司寿


しかし、UTF-8 バイト文字列で符号化した Unicode データではエラーとなります。

In [4]:
w = '寿司'
x = w.encode('utf-8')
y = x[::-1]
print(y.decode('utf-8'))  # 出力: b'\x81\x93\x81\x7a

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb8 in position 0: invalid start byte

-1 以外に負の増分が役に立つでしょうか？

In [6]:
x = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
print(x[::2])
print(x[::-2])

['a', 'c', 'e', 'g']
['h', 'f', 'd', 'b']


ここで、::2 は、先頭から2番目ずつの要素を選ぶことを意味します。
手が込んでいることに、::-2 は、末尾から2番目ずつの要素を逆順で選ぶことを意味します。

では、以下の例はどうでしょうか。

In [8]:
print(x[2::2])
print(x[-2::-2])
print(x[-2:2:-2])
print(x[2:2:-2])

['c', 'e', 'g']
['g', 'e', 'c', 'a']
['g', 'e']
[]


スライス構文のストライド部分が非常に紛らわしくなっています。

このような問題を避けるには、start や end のインデックスとストライドを一緒に使わないことです。

ストライドを使わなければならないときには、できる限り正の値にして、start と end のインデックスを省きます。

ストライドを start や end と一緒に使わなければならないときには、増分だけでの代入とスライスでの代入とに分けて使うように考えましょう。

In [10]:
y = x[::2]
z = y[1:-1]
print(y)
print(z)

['a', 'c', 'e', 'g']
['c', 'e']


ストライドしてスライスすると、データの浅い複製が余分に生じます。

最初の演算で、スライスの結果のサイズをできるだけ減らすようにすべきです。

プログラムに、2ステップにするだけの時間またはメモリの余裕がない場合には、組み込みモジュール itertools の islice メソッド（項目36）を使うことを検討しましょう。

これは、start、end、stride に負の値を許さないコードが明確になります。

## 覚えておくこと

- スライスで、start、end、stride をすべて指定すると、非常に読みにくい。
- スライスでは、正の増分値を start や end のどちらか一方のみと使う。可能な限り、負の増分値を使わない。
- 1つのスライスに、start、end、stride を一緒に使わない。3つのすべてが必要なときには、代入を2度使う（1つはスライスに、もう一つはストライドだけに）か、組み込みモジュール itertools の islice を使う。